# Data and problem overview

This is the pipeline's starting point. It loads paired implementations of the same programming problem (V0 and V1), checks that their code and measured energy/runtime/memory values are usable, and writes cleaned copies without changing the raw inputs. The project goal is to predict which implementation uses less energy from code features; these measured outcomes provide labels and later evaluation, not model inputs. Continue with Notebook 02 for exploratory comparisons, then Notebook 03 to build training features.

In [1]:
# Dependencies are installed in the selected project .venv.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Raw").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the project Data/Raw directory from this notebook's working directory.")
RAW_DIR = ROOT / "Data" / "Raw"
PROCESSED_DIR = ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PAIR_KEY = ["problem_id", "submission_id_v0", "submission_id_v1"]
REQUIRED_COLUMNS = [
    "user_id", "problem_id", "language", "submission_id_v0", "submission_id_v1",
    "code_v0_no_empty_lines", "code_v1_no_empty_lines", "result_energy_v0",
    "result_time_v0", "result_memory_v0", "result_energy_v1", "result_time_v1",
    "result_memory_v1",
]
RESOURCE_COLUMNS = [
    "result_energy_v0", "result_time_v0", "result_memory_v0",
    "result_energy_v1", "result_time_v1", "result_memory_v1",
]
RAW_FILES = {
    "validated": "public_validated_energy_results.jsonl",
    "significant": "public_significant_energy_diff.jsonl",
    "classified": "public_classified_smells.jsonl",
}

def load_and_clean_dataset(name, filename):
    path = RAW_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Required raw dataset is missing: {path}")
    frame = pd.read_json(path, lines=True)
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{filename} is missing required columns: {missing_columns}")

    before_rows = len(frame)
    missing_required = int(frame[REQUIRED_COLUMNS].isna().any(axis=1).sum())
    frame = frame.dropna(subset=REQUIRED_COLUMNS).copy()
    for column in RESOURCE_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    valid_metrics = np.isfinite(frame[RESOURCE_COLUMNS]).all(axis=1)
    valid_metrics &= frame[RESOURCE_COLUMNS].ge(0).all(axis=1)
    valid_code = frame["code_v0_no_empty_lines"].astype(str).str.strip().ne("")
    valid_code &= frame["code_v1_no_empty_lines"].astype(str).str.strip().ne("")
    invalid_measurements_or_code = int((~(valid_metrics & valid_code)).sum())
    frame = frame.loc[valid_metrics & valid_code].copy()

    duplicate_pairs = int(frame.duplicated(subset=PAIR_KEY, keep="first").sum())
    frame = frame.drop_duplicates(subset=PAIR_KEY, keep="first").reset_index(drop=True)

    # Optional annotation text is retained; nested values are serialized losslessly for Parquet.
    for column in frame.columns:
        if column in REQUIRED_COLUMNS or column in RESOURCE_COLUMNS:
            continue
        if frame[column].dtype == "object":
            frame[column] = frame[column].map(
                lambda value: "" if value is None or (isinstance(value, float) and np.isnan(value)) else value
            )
            if frame[column].map(lambda value: isinstance(value, (dict, list))).any():
                frame[column] = frame[column].map(
                    lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                )

    report = {
        "dataset": name,
        "raw_rows": before_rows,
        "rows_with_missing_required_values_removed": missing_required,
        "rows_with_invalid_metrics_or_empty_code_removed": invalid_measurements_or_code,
        "duplicate_pairs_removed": duplicate_pairs,
        "clean_rows": len(frame),
    }
    return frame, report

# Load each original source independently. Raw files remain unchanged.
df_full, validated_quality = load_and_clean_dataset("validated", RAW_FILES["validated"])
df_significant, significant_quality = load_and_clean_dataset("significant", RAW_FILES["significant"])
df_classified, classified_quality = load_and_clean_dataset("classified", RAW_FILES["classified"])
quality_report = pd.DataFrame([validated_quality, significant_quality, classified_quality])
print(quality_report.to_string(index=False))


    dataset  raw_rows  rows_with_missing_required_values_removed  rows_with_invalid_metrics_or_empty_code_removed  duplicate_pairs_removed  clean_rows
  validated     21428                                          0                                                0                        0       21428
significant      3000                                          0                                                0                        0        3000
 classified      3000                                          0                                                0                        0        3000


In [3]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 21428 entries, 0 to 21427
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_id                 21428 non-null  str    
 1   problem_id              21428 non-null  str    
 2   language                21428 non-null  str    
 3   submission_id_v0        21428 non-null  str    
 4   submission_id_v1        21428 non-null  str    
 5   code_v0_no_empty_lines  21428 non-null  str    
 6   code_v1_no_empty_lines  21428 non-null  str    
 7   result_energy_v0        21428 non-null  float64
 8   result_time_v0          21428 non-null  float64
 9   result_memory_v0        21428 non-null  int64  
 10  result_energy_v1        21428 non-null  float64
 11  result_time_v1          21428 non-null  float64
 12  result_memory_v1        21428 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 23.8 MB


In [4]:
# Display the first 5 rows of the dataset
print("\nFirst 5 Rows of the Dataset:")
display(df_full.head())


First 5 Rows of the Dataset:


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
0,u729133443,p03657,python,s252569931,s579542506,"print(((eval(input().replace(*"" *"")) % 3 % 2 *...","print(f""{eval(input().replace(*' *'))%3%2*'Imp...",512.63,4.123615,11188,540.49,4.877773,11144
1,u078349616,p02837,python,s425424388,s635043712,from itertools import product\n\nN = int(eval(...,N = int(eval(input()))\nXY = []\nfor i in rang...,603.91,4.851531,11192,580.07,4.693411,11280
2,u392029857,p03556,python,s273873551,s903554806,"n = int(eval(input()))\nfor i in range(n, 0, -...",n = int(eval(input()))\nprint((int(n**0.5) ** ...,562.08,4.771710,11168,407.27,3.290923,11188
3,u677096240,p02408,python,s826737558,s865280692,"n = int(input())\ncards, suits = [], [""S"", ""H""...",# ２次元配列バージョン\n# かつ、内包表記バージョン\nn = int(input())...,521.50,4.206890,11120,545.32,4.816959,11376
4,u624475441,p03290,python,s468722663,s706260018,"D, G = list(map(int, input().split()))\nP = []...","d, g = list(map(int, input().split()))\npc = [...",619.93,5.157960,11220,553.32,4.485817,11248


In [5]:
# Display the number of records and columns
print(f"\nNumber of records: {df_full.shape[0]}")
print(f"Number of columns: {df_full.shape[1]}")


Number of records: 21428
Number of columns: 13


### Data Dictoniary Table

| Column | Description | Data Type | Role | Summary / Range |
|---|---|---|---|---|
| `problem_id` | Programming task identifier | Categorical (object) | Grouping | 1,232 unique programming tasks |
| `code_v0_no_empty_lines` | Source code of implementation 0 | Text (object) | Input for feature extraction | 21,428 valid implementations |
| `code_v1_no_empty_lines` | Source code of implementation 1 | Text (object) | Input for feature extraction | 21,428 valid implementations |
| `result_energy_v0` | Energy consumption of implementation 0 | Numeric (float64) | Measurement | [352.01, 35307.72] Joules |
| `result_energy_v1` | Energy consumption of implementation 1 | Numeric (float64) | Measurement | [352.01, 35467.43] Joules |
| `result_time_v0` | Execution time of implementation 0 | Numeric (float64) | Measurement | [0.03, 110.87] seconds |
| `result_time_v1` | Execution time of implementation 1 | Numeric (float64) | Measurement | [0.03, 110.37] seconds |
| `result_memory_v0` | Memory usage of implementation 0 | Numeric (int64) | Measurement | [3072, 280388] KB |
| `result_memory_v1` | Memory usage of implementation 1 | Numeric (int64) | Measurement | [3072, 281864] KB |

# Data Quality Assements

In [6]:
# Data Quality Assessment: Check for Missing Values
print("\n--- Missing Values Assessment ---")
missing_values = df_full.isnull().sum()
missing_percentage = (df_full.isnull().sum() / len(df_full)) * 100

missing_info = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

# Filter to show only columns with missing values
missing_info = missing_info[missing_info['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)

if missing_info.empty:
    print("No missing values found in the dataset.")
else:
    print("Missing Values per Column:")
    display(missing_info)


--- Missing Values Assessment ---
No missing values found in the dataset.


In [7]:
# Data Quality Assessment: Check for Duplicate Records
print("\n--- Duplicate Records Assessment ---")
# A record is considered a duplicate if all its columns are identical.
duplicate_rows = df_full[df_full.duplicated()]

# Also check for 'logical duplicates' based on identifying columns
# What defines a unique experiment or comparison?
# A pair of submissions for a problem_id should be unique.
# The combination of problem_id, submission_id_v0, and submission_id_v1 should ideally be unique.
logical_duplicate_cols = ['problem_id', 'submission_id_v0', 'submission_id_v1']
duplicate_logical_entries = df_full[df_full.duplicated(subset=logical_duplicate_cols, keep=False)]

print(f"Total fully duplicate rows: {len(duplicate_rows)}")
if not duplicate_rows.empty:
    print("Sample fully duplicate rows:")
    display(duplicate_rows.head())

print(f"\nTotal logical duplicate entries (based on problem_id, submission_id_v0, submission_id_v1): {len(duplicate_logical_entries)}")
if not duplicate_logical_entries.empty:
    print("Sample logical duplicate entries:")
    display(duplicate_logical_entries.sort_values(by=logical_duplicate_cols).head())


--- Duplicate Records Assessment ---
Total fully duplicate rows: 0

Total logical duplicate entries (based on problem_id, submission_id_v0, submission_id_v1): 0


In [8]:
# Check for Invalid, Inconsistent, or Unexpected Values
print("\n--- Invalid/Inconsistent Values Assessment ---")

# Check 'language' column
print("\nUnique values in 'language' column:")
print(df_full['language'].value_counts())

# Check descriptive statistics for numerical resource columns
resource_cols = [
    'result_energy_v0', 'result_time_v0', 'result_memory_v0',
    'result_energy_v1', 'result_time_v1', 'result_memory_v1'
]

print("\nDescriptive Statistics for Resource Metrics:")
display(df_full[resource_cols].describe())


--- Invalid/Inconsistent Values Assessment ---

Unique values in 'language' column:
language
python    21428
Name: count, dtype: int64

Descriptive Statistics for Resource Metrics:


,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
count,21428.000000,21428.000000,2.142800e+04,21428.000000,21428.000000,2.142800e+04
mean,836.525068,9.815829,1.367307e+04,639.513029,7.758919,1.278427e+04
std,1898.938645,21.352216,3.529586e+04,1343.631862,14.829164,3.261878e+04
min,14.360000,0.179996,1.096400e+04,11.760000,0.146367,1.096400e+04
25%,354.680000,4.852967,1.117600e+04,354.110000,4.852669,1.117200e+04
50%,358.090000,4.855537,1.122800e+04,357.190000,4.855027,1.122400e+04
75%,370.970000,4.891021,1.132000e+04,364.700000,4.881466,1.131600e+04
max,42518.630000,490.812967,4.594516e+06,39617.780000,449.723400,4.548576e+06


In [9]:
# Outlier Detection for Numerical Measurements
print("\n--- Outlier Detection ---")

# Using IQR method to identify potential outliers
for col in resource_cols:
    Q1 = df_full[col].quantile(0.25)
    Q3 = df_full[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_full[(df_full[col] < lower_bound) | (df_full[col] > upper_bound)]
    print(f"\nPotential outliers in '{col}': {len(outliers)} records ({len(outliers)/len(df_full)*100:.2f}%) ")
    if not outliers.empty:
        # Display some sample outliers to understand their nature
        print("Sample Outliers (top 5 by value):")
        display(outliers.sort_values(by=col, ascending=False).head())



--- Outlier Detection ---

Potential outliers in 'result_energy_v0': 4874 records (22.75%) 
Sample Outliers (top 5 by value):


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
14314,u794173881,p03027,python,s719891765,s472042129,"class Combination:\n """"""\n 計算量：階乗・逆元テーブル...","class Combination:\n def __init__(self, n, ...",42518.63,490.812967,87592,9035.97,104.697780,95180
12545,u212437180,p02657,python,s112687013,s566354957,"from subprocess import run\n\nwith open(""./tmp...",from subprocess import run\nfrom os import pat...,35504.92,412.531598,233820,505.21,6.430518,11320
17709,u163783894,p02621,python,s958497875,s012776774,N = 10**7\nM = 10**5\narr = list(range(M))\n\n...,N = 10**7\n# M = 10**5\n# arr = list(range(M))...,34665.16,441.525855,12532,5150.80,65.963767,11032
19065,u268554510,p02769,python,s615148510,s072028091,"n, k = list(map(int, input().split()))\nk = mi...","n, k = list(map(int, input().split()))\nk = mi...",34579.40,400.185669,48192,14872.47,173.258019,24692
21234,u566872786,p00009,python,s751486986,s116067750,import sys\n\nMAX = 999999\n\n\ndef is_prime(n...,import sys\nimport time\n\nMAX = 999999\n\n\nd...,33574.27,424.378598,12260,17113.92,215.011464,12684



Potential outliers in 'result_time_v0': 5358 records (25.00%) 
Sample Outliers (top 5 by value):


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
14314,u794173881,p03027,python,s719891765,s472042129,"class Combination:\n """"""\n 計算量：階乗・逆元テーブル...","class Combination:\n def __init__(self, n, ...",42518.63,490.812967,87592,9035.97,104.697780,95180
17709,u163783894,p02621,python,s958497875,s012776774,N = 10**7\nM = 10**5\narr = list(range(M))\n\n...,N = 10**7\n# M = 10**5\n# arr = list(range(M))...,34665.16,441.525855,12532,5150.80,65.963767,11032
21234,u566872786,p00009,python,s751486986,s116067750,import sys\n\nMAX = 999999\n\n\ndef is_prime(n...,import sys\nimport time\n\nMAX = 999999\n\n\nd...,33574.27,424.378598,12260,17113.92,215.011464,12684
12545,u212437180,p02657,python,s112687013,s566354957,"from subprocess import run\n\nwith open(""./tmp...",from subprocess import run\nfrom os import pat...,35504.92,412.531598,233820,505.21,6.430518,11320
19065,u268554510,p02769,python,s615148510,s072028091,"n, k = list(map(int, input().split()))\nk = mi...","n, k = list(map(int, input().split()))\nk = mi...",34579.40,400.185669,48192,14872.47,173.258019,24692



Potential outliers in 'result_memory_v0': 1956 records (9.13%) 
Sample Outliers (top 5 by value):


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
8590,u314906167,p03241,python,s308381284,s693803332,"import sys\n\nn, m = list(map(int, input().spl...","import sys\nfrom math import sqrt\n\nn, m = li...",2831.47,32.771706,4594516,384.24,5.038367,11184
7190,u891635666,p02793,python,s642993086,s597786270,import collections\nimport sys\n\ninput = sys....,import collections\nimport sys\n\ninput = sys....,12267.05,140.774768,1263724,363.21,4.850667,11320
19113,u102461423,p02957,python,s142554424,s850468977,"import numpy as np\n\nA = np.arange(1 << 27, d...",import sys\n\nread = sys.stdin.buffer.read\nre...,7319.56,62.373502,549372,361.49,4.855669,11028
4061,u499381410,p03805,python,s527305718,s019519761,"from collections import defaultdict, deque, Co...","from collections import defaultdict, deque, Co...",998.37,11.947773,495604,849.38,11.054458,11400
20473,u255673886,p03805,python,s950656882,s894290346,import sys\n\nreadline = sys.stdin.buffer.read...,from collections import deque\nfrom heapq impo...,1976.17,23.380061,495020,1447.32,18.067074,54252


... (2 outputs left out)


Potential outliers in 'result_time_v1': 4422 records (20.64%) 
Sample Outliers (top 5 by value):


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
6597,u491763171,p00056,python,s580541092,s141714808,def sieve(n):\n primes = [True] * n\n pr...,def sieve(n):\n primes = [True] * n\n pr...,465.04,5.910289,11224,35570.69,449.723400,11272
10268,u748241164,p02862,python,s545981089,s073804557,"from math import factorial\n\nX, Y = list(map(...","def cmb(n, r, p):\n if (r < 0) or (n < r):\...",1201.80,15.048940,28068,39617.78,449.670499,244012
3578,u623819879,p02901,python,s825210706,s013046464,"from heapq import heappush, heappop\nfrom coll...","from subprocess import *\n\nwith open(""e.cpp"",...",465.68,5.979515,11196,26037.22,302.853724,193544
19828,u989345508,p02669,python,s077734530,s411123900,"check = dict()\nN, z, d = 0, [], 0\n# ttfは235、...","check = dict()\nN, z, d = 0, [], 0\n# ttfは235、...",20740.49,262.060295,12680,22610.30,283.541655,12884
11257,u021019433,p02642,python,s363241226,s536242029,N = 1000001\na = [0] * N\neval(input())\nfor x...,"N = 1000001\na = [0] * N\n_, s = open(0)\nfor ...",2646.25,32.839587,32292,21496.88,270.885346,16476



Potential outliers in 'result_memory_v1': 1397 records (6.52%) 
Sample Outliers (top 5 by value):


,user_id,problem_id,language,submission_id_v0,submission_id_v1,code_v0_no_empty_lines,code_v1_no_empty_lines,result_energy_v0,result_time_v0,result_memory_v0,result_energy_v1,result_time_v1,result_memory_v1
20304,u795630164,p03194,python,s085909586,s808239024,"import math\n\nN, P = list(map(int, input().sp...","import math\n\nN, P = list(map(int, input().sp...",381.61,5.224578,11264,9416.44,107.907244,4548576
19040,u623819879,p02793,python,s562953614,s517964009,"from heapq import heappush, heappop, heapify\n...","from heapq import heappush, heappop, heapify\n...",20365.28,225.058592,235616,3482.37,39.549089,501688
9913,u652656291,p03805,python,s057740404,s917484606,"n, m = [int(i) for i in input().split()]\nes =...","import itertools\n\nN, M = list(map(int, input...",356.08,4.851697,11316,905.53,10.894610,494436
7056,u123756661,p03805,python,s861018210,s184220268,"import itertools\n\nn, m = list(map(int, input...","import itertools\n\nn, m = list(map(int, input...",418.88,5.507786,51580,1091.06,12.975761,494416
10814,u886747123,p02990,python,s899606605,s579144050,# D - Blue and Red Balls\n# 青いボールを並べる→赤いボールを青い...,"# D - Blue and Red Balls\nN, K = list(map(int,...",1418.75,16.804923,293408,1419.08,16.837662,293396


### Data Validation: Checking for Negative, Zero, and Malformed Measurements

In [10]:
import pandas as pd

# 1. Check for negative measurements
print("=== 1. Checking for Negative Values ===")
neg_energy_v0 = (df_full['result_energy_v0'] < 0).sum()
neg_energy_v1 = (df_full['result_energy_v1'] < 0).sum()
neg_time_v0 = (df_full['result_time_v0'] < 0).sum()
neg_time_v1 = (df_full['result_time_v1'] < 0).sum()
neg_mem_v0 = (df_full['result_memory_v0'] < 0).sum()
neg_mem_v1 = (df_full['result_memory_v1'] < 0).sum()

print(f"Negative Energy (v0): {neg_energy_v0} | (v1): {neg_energy_v1}")
print(f"Negative Time (v0): {neg_time_v0} | (v1): {neg_time_v1}")
print(f"Negative Memory (v0): {neg_mem_v0} | (v1): {neg_mem_v1}")

# 2. Check for zero measurements
print("\n=== 2. Checking for Zero Values ===")
zero_energy_v0 = (df_full['result_energy_v0'] == 0).sum()
zero_energy_v1 = (df_full['result_energy_v1'] == 0).sum()
zero_time_v0 = (df_full['result_time_v0'] == 0).sum()
zero_time_v1 = (df_full['result_time_v1'] == 0).sum()
zero_mem_v0 = (df_full['result_memory_v0'] == 0).sum()
zero_mem_v1 = (df_full['result_memory_v1'] == 0).sum()

print(f"Zero Energy (v0): {zero_energy_v0} | (v1): {zero_energy_v1}")
print(f"Zero Time (v0): {zero_time_v0} | (v1): {zero_time_v1}")
print(f"Zero Memory (v0): {zero_mem_v0} | (v1): {zero_mem_v1}")

# 3. Check for invalid or empty source code
print("\n=== 3. Checking for Empty/Malformed Code ===")
empty_code_v0 = df_full['code_v0_no_empty_lines'].apply(lambda x: str(x).strip() == '').sum()
empty_code_v1 = df_full['code_v1_no_empty_lines'].apply(lambda x: str(x).strip() == '').sum()
print(f"Empty/Whitespace-only code (v0): {empty_code_v0}")
print(f"Empty/Whitespace-only code (v1): {empty_code_v1}")

# 4. Check for incomplete or unmatched pairs
print("\n=== 4. Checking for Unmatched Pairs ===")
missing_sub_v0 = df_full['submission_id_v0'].isnull().sum()
missing_sub_v1 = df_full['submission_id_v1'].isnull().sum()
print(f"Missing submission identifier (v0): {missing_sub_v0}")
print(f"Missing submission identifier (v1): {missing_sub_v1}")

=== 1. Checking for Negative Values ===
Negative Energy (v0): 0 | (v1): 0
Negative Time (v0): 0 | (v1): 0
Negative Memory (v0): 0 | (v1): 0

=== 2. Checking for Zero Values ===
Zero Energy (v0): 0 | (v1): 0
Zero Time (v0): 0 | (v1): 0
Zero Memory (v0): 0 | (v1): 0

=== 3. Checking for Empty/Malformed Code ===
Empty/Whitespace-only code (v0): 0
Empty/Whitespace-only code (v1): 0

=== 4. Checking for Unmatched Pairs ===
Missing submission identifier (v0): 0
Missing submission identifier (v1): 0


### Check for Non-ASCII or Unexpected Characters in Source Code

In [11]:
def check_non_ascii(code_str):
    if not isinstance(code_str, str):
        return []
    # Report characters outside printable ASCII while keeping the diagnostic Windows-console safe.
    return [char for char in code_str if ord(char) > 127 or (ord(char) < 32 and char not in "\n\r\t")]

df_full["non_ascii_v0"] = df_full["code_v0_no_empty_lines"].apply(check_non_ascii)
df_full["non_ascii_v1"] = df_full["code_v1_no_empty_lines"].apply(check_non_ascii)
rows_with_non_ascii_v0 = df_full[df_full["non_ascii_v0"].map(len) > 0]
rows_with_non_ascii_v1 = df_full[df_full["non_ascii_v1"].map(len) > 0]
print(f"Rows with unusual characters in v0 code: {len(rows_with_non_ascii_v0)}")
print(f"Rows with unusual characters in v1 code: {len(rows_with_non_ascii_v1)}")

if len(rows_with_non_ascii_v0) > 0 or len(rows_with_non_ascii_v1) > 0:
    print("\n--- Sample records with unusual characters ---")
    unique_chars = set()
    for chars in df_full["non_ascii_v0"]:
        unique_chars.update(chars)
    for chars in df_full["non_ascii_v1"]:
        unique_chars.update(chars)
    print(f"Unique non-standard characters found: {ascii(list(unique_chars)[:50])}")
    samples = df_full[
        (df_full["non_ascii_v0"].map(len) > 0) | (df_full["non_ascii_v1"].map(len) > 0)
    ].head(5)
    samples["code_v0_no_empty_lines"] = samples["code_v0_no_empty_lines"].map(lambda code: ascii(code[:300]))
    samples["code_v1_no_empty_lines"] = samples["code_v1_no_empty_lines"].map(lambda code: ascii(code[:300]))
    samples["non_ascii_v0"] = samples["non_ascii_v0"].map(ascii)
    samples["non_ascii_v1"] = samples["non_ascii_v1"].map(ascii)
    display(samples[[
        "problem_id", "code_v0_no_empty_lines", "non_ascii_v0",
        "code_v1_no_empty_lines", "non_ascii_v1",
    ]])
df_full = df_full.drop(columns=["non_ascii_v0", "non_ascii_v1"])


Rows with unusual characters in v0 code: 2560
Rows with unusual characters in v1 code: 2631

--- Sample records with unusual characters ---
Unique non-standard characters found: ['\u821e', '\u642c', '\u6b62', '\u91e3', '\u30cd', '\u5e7c', '\u30e2', '\u7bb1', '\u753b', '\u53ca', '\uff59', '\x01', '\u5c3d', '\u5076', '\u5e8f', '\u6734', '\uff16', '\u5f8c', '\u7b56', '\u5c4a', '\u4e8c', '\u5165', '\u9762', '\u5f71', '\u5f3e', '\u5de5', '\uff1d', '\u70ba', '\u4e0d', '\u8349', '\u517c', '\u9577', '\u4f53', '\u540d', '\u584a', '\u5270', '\xa3', '\u307c', '\u7d66', '\xfc', '\u8ef8', '\u8db3', '\u523a', '\u6cc1', '\uff3c', '\u30e9', '\u7d4c', '\uff09', '\u7bc9', '\u5374']


,problem_id,code_v0_no_empty_lines,non_ascii_v0,code_v1_no_empty_lines,non_ascii_v1
1,p02837,'from itertools import product\n\nN = int(eval...,"['\u4e0d', '\u6b63', '\u76f4', '\u6b63', '\u76...",'N = int(eval(input()))\nXY = []\nfor i in ran...,[]
3,p02408,"'n = int(input())\ncards, suits = [], [""S"", ""H...",[],'# \uff12\u6b21\u5143\u914d\u5217\u30d0\u30fc\...,"['\uff12', '\u6b21', '\u5143', '\u914d', '\u52..."
12,p02412,'while 1:\n comb = []\n count = 0\n i...,[],"'# coding: UTF-8\ndef find_comb(n, x):\n co...","['\u63a2', '\u7d22', '\u2190', '\u304c', '\u30..."
17,p03103,"'from functools import reduce\n\nN, M = list(m...",[],'from functools import reduce\n\n# \u5165\u529...,"['\u5165', '\u529b', '\u672c', '\u3042', '\u30..."
35,p02660,'# \u7d04\u6570\ndef make_divisors(n): # \u7d...,"['\u7d04', '\u6570', '\u7d04', '\u6570', '\u30...",'# \u7d04\u6570\u306f\u4f7f\u308f\u305a\u3001\...,"['\u7d04', '\u6570', '\u306f', '\u4f7f', '\u30..."


### Identify algorithms and comparable implementations

In [12]:
# Identify all algorithms, programming tasks, or implementation categories
print("\n--- Algorithm/Task Identification ---")

# Group by 'problem_id' to identify unique tasks using df_full
unique_problems = df_full['problem_id'].nunique()
print(f"Number of unique programming problems (tasks): {unique_problems}")

# Count records per problem_id
records_per_problem = df_full['problem_id'].value_counts()
print("\nTop 10 problems by number of records:")
display(records_per_problem.head(10))

print("\nBottom 10 problems by number of records:")
display(records_per_problem.tail(10))


--- Algorithm/Task Identification ---
Number of unique programming problems (tasks): 1232

Top 10 problems by number of records:


problem_id
p03160    268
p02713    193
p02720    188
p03162    176
p03835    175
p03329    174
p02659    174
p03078    173
p03478    160
p02695    160
Name: count, dtype: int64


Bottom 10 problems by number of records:


problem_id
p02377    1
p03080    1
p01130    1
p03364    1
p02360    1
p03975    1
p01267    1
p03653    1
p02565    1
p03019    1
Name: count, dtype: int64

In [13]:
# Determine whether the dataset contains multiple implementations of the same task.
print("\n--- Multiple Implementations per Task ---")

# Extract all submission IDs for v0 and v1
all_submission_ids = pd.concat([
    df_full[['problem_id', 'submission_id_v0']].rename(columns={'submission_id_v0': 'submission_id'}),
    df_full[['problem_id', 'submission_id_v1']].rename(columns={'submission_id_v1': 'submission_id'})
])

# Count distinct submissions per problem_id
distinct_implementations_per_problem = all_submission_ids.groupby('problem_id')['submission_id'].nunique()

print("Distribution of distinct implementations per problem_id:")
print(distinct_implementations_per_problem.value_counts().sort_index())

# Identify problems with at least 3 distinct implementations
problems_with_3_plus_impl = distinct_implementations_per_problem[distinct_implementations_per_problem >= 3]

print(f"\nNumber of problems with at least 3 distinct implementations: {len(problems_with_3_plus_impl)}")
if not problems_with_3_plus_impl.empty:
    print("Sample problems with 3+ distinct implementations (problem_id and count):")
    display(problems_with_3_plus_impl.head())


--- Multiple Implementations per Task ---
Distribution of distinct implementations per problem_id:
submission_id
2      262
3       14
4      130
5       13
6       79
      ... 
340      1
341      1
365      1
373      1
521      1
Name: count, Length: 176, dtype: int64

Number of problems with at least 3 distinct implementations: 970
Sample problems with 3+ distinct implementations (problem_id and count):


problem_id
p00001    14
p00002    19
p00003    16
p00004     8
p00006     6
Name: submission_id, dtype: int64

In [14]:
# How many different problems?
print("Unique problems:", df_full["problem_id"].nunique())

# How many records per problem?
print(df_full["problem_id"].value_counts().head(20))

# What classifications exist?
if "final_classification" in df_full.columns:
    print(df_full["final_classification"].value_counts())
elif 'df_sig' in globals() and "final_classification" in df_sig.columns:
    print(df_sig["final_classification"].value_counts())
else:
    print("Column 'final_classification' not found in current main DataFrame.")

# Display one example pair of code submissions
row = df_full.iloc[0]

print("Problem ID:", row["problem_id"])
print("Language:", row["language"])

print("\nVersion 0:\n", row["code_v0_no_empty_lines"])
print("\nVersion 1:\n", row["code_v1_no_empty_lines"])

print("\nEnergy v0:", row["result_energy_v0"])
print("Energy v1:", row["result_energy_v1"])

Unique problems: 1232
problem_id
p03160    268
p02713    193
p02720    188
p03162    176
p03835    175
p03329    174
p02659    174
p03078    173
p03478    160
p02695    160
p03805    153
p03013    153
p02837    138
p02983    133
p02726    132
p03212    130
p03476    130
p03031    129
p03221    127
p02996    125
Name: count, dtype: int64
Column 'final_classification' not found in current main DataFrame.
Problem ID: p03657
Language: python

Version 0:
 print(((eval(input().replace(*" *")) % 3 % 2 * "Imp" or "P") + "ossible"))


Version 1:
 print(f"{eval(input().replace(*' *'))%3%2*'Imp'or'P'}ossible")


Energy v0: 512.63
Energy v1: 540.49


# Mapping Problem ID to Problem names
To better understand the dataset

In [15]:
import pandas as pd

# The raw files have no task-title column; use observed problem IDs without a network dependency.
problems = pd.DataFrame({"id": sorted(df_full["problem_id"].dropna().unique())})
problems["slug_name"] = problems["id"]
print(f"Using {len(problems)} observed problem IDs; no external metadata download is required.")


Using 1232 observed problem IDs; no external metadata download is required.


In [16]:
print(problems.columns.tolist())

['id', 'slug_name']


In [17]:
print(problems.head(10).to_string())

       id slug_name
0  p00001    p00001
1  p00002    p00002
2  p00003    p00003
3  p00004    p00004
4  p00005    p00005
5  p00006    p00006
6  p00007    p00007
7  p00008    p00008
8  p00009    p00009
9  p00010    p00010


In [18]:
print("\n--- Merging problem names ---")
if 'problems' in globals() and not problems.empty:
    df_full = pd.merge(df_full, problems[['id', 'slug_name']], left_on='problem_id', right_on='id', how='left')
    df_full = df_full.rename(columns={'slug_name': 'problem_name'}).drop(columns=['id'])
else:
    print("HuggingFace dataset not loaded. Creating placeholder problem names.")
    df_full['problem_name'] = df_full['problem_id']

print("problem_name column processed. Displaying first few rows:")
display(df_full[['problem_id', 'problem_name']].head())


--- Merging problem names ---
problem_name column processed. Displaying first few rows:


,problem_id,problem_name
0,p03657,p03657
1,p02837,p02837
2,p03556,p03556
3,p02408,p02408
4,p03290,p03290


In [19]:
# Verify that problem_name has been added to the DataFrame
print("\nUpdated DataFrame columns:")
print(df_full.columns.tolist())


Updated DataFrame columns:
['user_id', 'problem_id', 'language', 'submission_id_v0', 'submission_id_v1', 'code_v0_no_empty_lines', 'code_v1_no_empty_lines', 'result_energy_v0', 'result_time_v0', 'result_memory_v0', 'result_energy_v1', 'result_time_v1', 'result_memory_v1', 'problem_name']


In [20]:
import pandas as pd

if 'problems' in globals() and not problems.empty:
    problem_table = problems[
        ["id", "slug_name"]
    ].drop_duplicates().rename(columns={"id": "problem_id", "slug_name": "problem_name"})
else:
    print("Using problem_id as fallback for problem_table.")
    problem_table = pd.DataFrame({
        'problem_id': df_full['problem_id'].unique(),
        'problem_name': df_full['problem_id'].unique()
    })

implementations_v0 = df_full[['problem_id', 'submission_id_v0', 'language']].rename(columns={'submission_id_v0': 'submission_id'})
implementations_v1 = df_full[['problem_id', 'submission_id_v1', 'language']].rename(columns={'submission_id_v1': 'submission_id'})
implementation_table = pd.concat([implementations_v0, implementations_v1]).drop_duplicates()

print("problem_table created successfully with columns:", problem_table.columns.tolist())
print("implementation_table created successfully with columns:", implementation_table.columns.tolist())

problem_table created successfully with columns: ['problem_id', 'problem_name']
implementation_table created successfully with columns: ['problem_id', 'submission_id', 'language']


In [21]:
# Persist cleaned copies; original JSONL files in Data/Raw are never modified.
processed_datasets = {
    "validated_energy_results": df_full,
    "significant_energy_diff": df_significant,
    "classified_smells": df_classified,
}
for name, frame in processed_datasets.items():
    output_path = PROCESSED_DIR / f"{name}.parquet"
    frame.to_parquet(output_path, index=False)
    print(f"Saved {len(frame):,} rows to {output_path.relative_to(ROOT)}")
quality_report.to_csv(PROCESSED_DIR / "data_quality_report.csv", index=False)


Saved 21,428 rows to Data\Processed\validated_energy_results.parquet
Saved 3,000 rows to Data\Processed\significant_energy_diff.parquet
Saved 3,000 rows to Data\Processed\classified_smells.parquet
